# Phân tích Thực nghiệm Tự động Co giãn Dịch vụ Web (HPA CPU vs. KEDA Custom Metrics)

Notebook này tổng hợp dữ liệu đo thô (`metadata.json`, `k6-summary.json`, `prometheus_timeseries.csv`) từ 3 chính sách co giãn (`baseline-cpu`, `custom-unoptimized`, `custom-optimized`) trên các kịch bản tải (`step`, `spike`, `periodic`, `fault`), tính toán đầy đủ các chỉ số hiệu năng - chi phí - dao động, xuất bảng đối chiếu trước/sau tối ưu và vẽ các biểu đồ chuẩn học thuật (có đầy đủ tên trục, đơn vị, điều kiện đo và số lần lặp $n$).

> **Nguyên tắc toàn vẹn dữ liệu:** Chỉ phân tích dữ liệu đo thực tế trong thư mục `data/`. Các chỉ số không thu thập được sẽ giữ nguyên giá trị `NaN` (không tự ý gán bằng `0`).

## 1. Khởi tạo Môi trường & Cấu hình Đường dẫn

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

POLICIES = ["baseline-cpu", "custom-unoptimized", "custom-optimized"]
POLICY_LABELS = {
    "baseline-cpu": "HPA CPU (Baseline)",
    "custom-unoptimized": "KEDA Raw (Unoptimized)",
    "custom-optimized": "KEDA Tuned (Optimized)",
}
POLICY_COLORS = {
    "baseline-cpu": "#4C72B0",
    "custom-unoptimized": "#DD8452",
    "custom-optimized": "#55A868",
}
MAX_CONCURRENT_WORKERS_PER_POD = 5
FLAPPING_WINDOW_SECONDS = 90.0

print("Data root   :", DATA.resolve())
print("Figures root:", FIGURES.resolve())

## 2. Thu thập Dữ liệu Đo Thô & Tính toán Chỉ số Thực nghiệm

Các chỉ số được tính cho từng lượt chạy (run) gồm:
- **Chất lượng dịch vụ (QoS):** `p95_ms` (độ trễ toàn trình p95), `queue_wait_p95_ms` (độ trễ chờ hàng đợi p95), `throughput_rps` (thông lượng), `error_rate` (tỷ lệ lỗi), `overload_503` (số yêu cầu bị từ chối HTTP 503).
- **Bão hòa & Tài nguyên (Saturation & Resources):** `cpu_cores_avg`, `memory_mib_avg`, `queue_length_avg`, `queue_length_peak`, `in_flight_avg`, `worker_saturation_pct` (tỷ lệ sử dụng worker đồng thời so với giới hạn 5 workers/Pod).
- **Động học Co giãn & Dao động (Scaling Dynamics & Flapping):** `replicas_avg`, `replicas_peak`, `scale_out_s` (thời gian phản ứng mở rộng bản sao đầu tiên), `scale_in_s` (thời gian thu hồi bản sao về mức nền), `replica_changes` (tổng số lần đổi số bản sao), `flapping_count` (số lần đảo chiều tăng $\leftrightarrow$ giảm liên tiếp trong cửa sổ $\le 90\text{s}$).
- **Chi phí Tài nguyên (Resource Cost):**
  $$\text{ReplicaSeconds} = \sum_i \text{Replicas}_i \times \Delta t_i, \quad \text{Cost per Request} = \frac{\text{ReplicaSeconds}}{\text{Successful Requests}}$$

In [ ]:
def k6_stat(summary: dict, metric: str, key: str, fallback: float = float("nan")) -> float:
    block = summary.get("metrics", {}).get(metric, {})
    val = block.get("values", {}).get(key, fallback)
    try:
        return float(val)
    except (TypeError, ValueError):
        return fallback


def compute_flapping_count(timestamps: np.ndarray, replicas: np.ndarray, window_s: float = FLAPPING_WINDOW_SECONDS) -> int:
    """Count rapid direction reversals (up->down or down->up) occurring within `window_s` seconds."""
    if len(replicas) < 3:
        return 0
    diffs = np.diff(replicas)
    change_idx = np.flatnonzero(diffs != 0)
    if len(change_idx) < 2:
        return 0
    flaps = 0
    for k in range(1, len(change_idx)):
        prev_i = change_idx[k - 1]
        curr_i = change_idx[k]
        prev_dir = np.sign(diffs[prev_i])
        curr_dir = np.sign(diffs[curr_i])
        dt = float(timestamps[curr_i + 1] - timestamps[prev_i + 1])
        if prev_dir != curr_dir and dt <= window_s:
            flaps += 1
    return int(flaps)


def format_condition_str(df_sub: pd.DataFrame) -> str:
    """Build human-readable experimental condition string for chart subtitles."""
    if df_sub.empty:
        return "No data"
    first = df_sub.iloc[0]
    endpoint = first.get("endpoint", "unknown")
    rate = first.get("rate", float("nan"))
    max_vus = first.get("max_vus", float("nan"))
    sample_s = first.get("sample_seconds", 5)
    parts = [f"Endpoint={endpoint}"]
    if pd.notna(rate):
        parts.append(f"BaseRate={int(rate)} req/s")
    if pd.notna(max_vus):
        parts.append(f"MaxVUs={int(max_vus)}")
    if endpoint == "cpu" and pd.notna(first.get("iterations")):
        parts.append(f"Iterations={int(first['iterations']):,}")
    elif endpoint == "queue":
        if pd.notna(first.get("work_time")):
            parts.append(f"WorkTime={first['work_time']}s")
        if pd.notna(first.get("max_queue_limit")):
            parts.append(f"QueueLimit={int(first['max_queue_limit'])}")
    parts.append(f"Step={int(sample_s)}s")
    return ", ".join(parts)


records = []
for policy in POLICIES:
    for metadata_path in sorted((DATA / policy).glob("*/metadata.json")):
        folder = metadata_path.parent
        summary_path = folder / "k6-summary.json"
        if not summary_path.exists():
            print("Missing k6 summary (skipped):", folder.name)
            continue
        try:
            meta = json.loads(metadata_path.read_text(encoding="utf-8-sig"))
            summary = json.loads(summary_path.read_text(encoding="utf-8-sig"))
        except (ValueError, OSError) as err:
            print("Invalid run:", folder, err)
            continue

        duration = float(meta.get("ended_unix", 0) - meta.get("started_unix", 0))
        total_reqs = k6_stat(summary, "http_reqs", "count")
        error_rate = k6_stat(summary, "app_error_rate", "rate")
        if math.isnan(error_rate):
            error_rate = k6_stat(summary, "http_req_failed", "rate")
        successful_reqs = total_reqs * (1.0 - (error_rate if pd.notna(error_rate) else 0.0)) if pd.notna(total_reqs) else float("nan")
        queue_wait_p95_s = k6_stat(summary, "queue_wait_seconds", "p(95)")

        row = {
            "policy": meta.get("policy", policy),
            "scenario": meta.get("scenario", "unknown"),
            "endpoint": meta.get("endpoint", "unknown"),
            "repeat": int(meta.get("repeat", 1)),
            "run": folder.name,
            "rate": meta.get("rate", float("nan")),
            "max_vus": meta.get("max_vus", float("nan")),
            "work_time": meta.get("work_time", float("nan")),
            "iterations": meta.get("iterations", float("nan")),
            "max_queue_limit": meta.get("max_queue_limit", meta.get("queue_limit", float("nan"))),
            "sample_seconds": meta.get("sample_seconds", 5),
            "load_balanced": bool(meta.get("load_balanced", False)),
            "safety_stop": meta.get("safety_stop"),
            "duration_s": duration,
            "requests": total_reqs,
            "successful_requests": successful_reqs,
            "throughput_rps": k6_stat(summary, "http_reqs", "rate"),
            "p95_ms": k6_stat(summary, "http_req_duration", "p(95)"),
            "queue_wait_p95_ms": queue_wait_p95_s * 1000.0 if pd.notna(queue_wait_p95_s) else float("nan"),
            "error_rate": error_rate,
            "error_rate_pct": error_rate * 100.0 if pd.notna(error_rate) else float("nan"),
            "dropped_iterations": k6_stat(summary, "dropped_iterations", "count", 0.0),
            "overload_503": k6_stat(summary, "overload_503_total", "count", 0.0),
            "cpu_cores_avg": float("nan"),
            "memory_mib_avg": float("nan"),
            "queue_length_avg": float("nan"),
            "queue_length_peak": float("nan"),
            "in_flight_avg": float("nan"),
            "worker_saturation_pct": float("nan"),
            "replicas_peak": float("nan"),
            "replicas_avg": float("nan"),
            "replica_seconds": float("nan"),
            "cost_per_req_replica_s": float("nan"),
            "cost_per_1k_req_replica_s": float("nan"),
            "scale_out_s": float("nan"),
            "scale_in_s": float("nan"),
            "replica_changes": float("nan"),
            "flapping_count": float("nan"),
        }

        metrics_file = folder / "prometheus_timeseries.csv"
        if metrics_file.exists():
            ts = pd.read_csv(metrics_file)
            if not ts.empty:
                for col in ["timestamp", "cpu_cores", "memory_bytes", "replicas_available", "replicas_desired", "queue_length", "in_flight", "app_rps"]:
                    if col in ts:
                        ts[col] = pd.to_numeric(ts[col], errors="coerce")
                ts = ts.sort_values("timestamp").reset_index(drop=True)

                # Filter to the active experiment window [started_unix, ended_unix] when available
                t_start = meta.get("started_unix")
                t_end = meta.get("ended_unix")
                ts_win = ts[(ts["timestamp"] >= t_start) & (ts["timestamp"] <= t_end)] if (t_start and t_end) else ts
                if ts_win.empty:
                    ts_win = ts

                if "cpu_cores" in ts_win:
                    row["cpu_cores_avg"] = ts_win["cpu_cores"].mean()
                if "memory_bytes" in ts_win:
                    row["memory_mib_avg"] = ts_win["memory_bytes"].mean() / (1024 ** 2)
                if "queue_length" in ts_win and ts_win["queue_length"].notna().any():
                    row["queue_length_avg"] = ts_win["queue_length"].mean()
                    row["queue_length_peak"] = ts_win["queue_length"].max()
                if "in_flight" in ts_win and ts_win["in_flight"].notna().any():
                    row["in_flight_avg"] = ts_win["in_flight"].mean()

                replica_col = (
                    "replicas_available"
                    if "replicas_available" in ts and ts["replicas_available"].notna().any()
                    else "replicas_desired"
                )
                if replica_col in ts and ts[replica_col].notna().any():
                    valid = ts[["timestamp", replica_col]].dropna().sort_values("timestamp")
                    valid_win = valid[(valid["timestamp"] >= t_start) & (valid["timestamp"] <= t_end)] if (t_start and t_end) else valid
                    if valid_win.empty:
                        valid_win = valid

                    row["replicas_peak"] = valid_win[replica_col].max()
                    row["replicas_avg"] = valid_win[replica_col].mean()
                    row["replica_changes"] = int(valid[replica_col].diff().fillna(0).ne(0).sum())
                    row["flapping_count"] = compute_flapping_count(
                        valid["timestamp"].to_numpy(), valid[replica_col].to_numpy()
                    )

                    # Compute ReplicaSeconds = sum(ReplicaCount_i * dt_i) over the experiment window
                    step_s = float(meta.get("sample_seconds", 5))
                    if len(valid_win) > 1:
                        dts = np.diff(valid_win["timestamp"].to_numpy(), append=valid_win["timestamp"].iloc[-1] + step_s)
                        dts = np.clip(dts, 0.0, step_s * 3.0)
                        replica_sec = float(np.sum(valid_win[replica_col].to_numpy() * dts))
                    else:
                        replica_sec = float(valid_win[replica_col].iloc[0] * max(duration, step_s))
                    row["replica_seconds"] = replica_sec

                    if pd.notna(successful_reqs) and successful_reqs > 0:
                        row["cost_per_req_replica_s"] = replica_sec / successful_reqs
                        row["cost_per_1k_req_replica_s"] = (replica_sec / successful_reqs) * 1000.0

                    # Worker saturation (%) = in_flight / (replicas * MAX_CONCURRENT_WORKERS_PER_POD) * 100
                    if "in_flight" in ts_win and ts_win["in_flight"].notna().any():
                        merged_cap = ts_win[[replica_col, "in_flight"]].dropna()
                        merged_cap = merged_cap[merged_cap[replica_col] > 0]
                        if not merged_cap.empty:
                            sat = (merged_cap["in_flight"] / (merged_cap[replica_col] * MAX_CONCURRENT_WORKERS_PER_POD)) * 100.0
                            row["worker_saturation_pct"] = float(sat.mean())

                    origin = float(meta.get("started_unix", valid["timestamp"].iloc[0]))
                    baseline = valid.loc[valid["timestamp"] <= origin, replica_col]
                    initial = baseline.iloc[-1] if not baseline.empty else valid[replica_col].iloc[0]
                    up = valid[(valid["timestamp"] >= origin) & (valid[replica_col] > initial)]
                    if not up.empty:
                        row["scale_out_s"] = float(up["timestamp"].iloc[0] - origin)
                        after_up = valid[valid["timestamp"] > up["timestamp"].iloc[0]]
                        down = after_up[after_up[replica_col] <= initial]
                        if not down.empty:
                            row["scale_in_s"] = float(down["timestamp"].iloc[0] - up["timestamp"].iloc[0])

        records.append(row)

runs = pd.DataFrame.from_records(records)
if runs.empty:
    print("CHƯA CÓ DỮ LIỆU THỰC NGHIỆM. Chạy scripts/run-experiment.ps1 hoặc scripts/run-all-experiments.ps1 trước.")
else:
    runs = runs.sort_values(["scenario", "endpoint", "policy", "repeat"]).reset_index(drop=True)
    display(runs)
    runs.to_csv(DATA / "comparison-runs.csv", index=False)
    print(f"Đã lưu {len(runs)} lượt chạy vào: {DATA / 'comparison-runs.csv'}")

## 3. Bảng Tổng hợp theo Kịch bản & So sánh Trước / Sau Tối ưu

Tổng hợp trung bình (mean) và độ lệch chuẩn (std) qua các lần lặp ($n = \text{repeats}$), kèm bảng đối chiếu mức độ cải thiện ($\Delta \%$) giữa các chính sách.

In [ ]:
if not runs.empty:
    aggregate = (
        runs.groupby(["scenario", "endpoint", "policy"], dropna=False)
        .agg(
            repeats=("run", "count"),
            rate=("rate", "first"),
            max_vus=("max_vus", "first"),
            work_time=("work_time", "first"),
            iterations=("iterations", "first"),
            max_queue_limit=("max_queue_limit", "first"),
            sample_seconds=("sample_seconds", "first"),
            p95_ms=("p95_ms", "mean"),
            p95_ms_std=("p95_ms", "std"),
            queue_wait_p95_ms=("queue_wait_p95_ms", "mean"),
            throughput_rps=("throughput_rps", "mean"),
            error_rate_pct=("error_rate_pct", "mean"),
            overload_503=("overload_503", "mean"),
            cpu_cores_avg=("cpu_cores_avg", "mean"),
            memory_mib_avg=("memory_mib_avg", "mean"),
            worker_saturation_pct=("worker_saturation_pct", "mean"),
            queue_length_avg=("queue_length_avg", "mean"),
            queue_length_peak=("queue_length_peak", "max"),
            replicas_avg=("replicas_avg", "mean"),
            replicas_peak=("replicas_peak", "max"),
            replica_seconds=("replica_seconds", "mean"),
            cost_per_1k_req_replica_s=("cost_per_1k_req_replica_s", "mean"),
            scale_out_s=("scale_out_s", "mean"),
            scale_in_s=("scale_in_s", "mean"),
            replica_changes=("replica_changes", "mean"),
            flapping_count=("flapping_count", "mean"),
            dropped_iterations=("dropped_iterations", "sum"),
        )
        .reset_index()
    )
    # Order policies consistently
    aggregate["policy_order"] = aggregate["policy"].map({p: i for i, p in enumerate(POLICIES)}).fillna(99)
    aggregate = aggregate.sort_values(["scenario", "endpoint", "policy_order"]).drop(columns=["policy_order"]).reset_index(drop=True)
    display(aggregate)
    aggregate.to_csv(DATA / "comparison-aggregated.csv", index=False)

    # Build Before-vs-After optimization comparison table when multiple policies exist for a scenario
    comparison_rows = []
    for (scenario, endpoint), grp in aggregate.groupby(["scenario", "endpoint"]):
        by_pol = grp.set_index("policy")
        if "custom-optimized" in by_pol.index:
            opt = by_pol.loc["custom-optimized"]
            for ref_pol in ["baseline-cpu", "custom-unoptimized"]:
                if ref_pol in by_pol.index:
                    ref = by_pol.loc[ref_pol]
                    def pct_change(new_val, old_val):
                        if pd.isna(new_val) or pd.isna(old_val) or old_val == 0:
                            return float("nan")
                        return ((new_val - old_val) / abs(old_val)) * 100.0
                    comparison_rows.append({
                        "scenario": scenario,
                        "endpoint": endpoint,
                        "comparison": f"custom-optimized vs. {ref_pol}",
                        "repeats_opt": int(opt["repeats"]),
                        "repeats_ref": int(ref["repeats"]),
                        "p95_ms_ref": ref["p95_ms"],
                        "p95_ms_opt": opt["p95_ms"],
                        "p95_ms_delta_pct": pct_change(opt["p95_ms"], ref["p95_ms"]),
                        "error_rate_pct_ref": ref["error_rate_pct"],
                        "error_rate_pct_opt": opt["error_rate_pct"],
                        "scale_out_s_ref": ref["scale_out_s"],
                        "scale_out_s_opt": opt["scale_out_s"],
                        "cost_per_1k_req_ref": ref["cost_per_1k_req_replica_s"],
                        "cost_per_1k_req_opt": opt["cost_per_1k_req_replica_s"],
                        "cost_delta_pct": pct_change(opt["cost_per_1k_req_replica_s"], ref["cost_per_1k_req_replica_s"]),
                        "flapping_ref": ref["flapping_count"],
                        "flapping_opt": opt["flapping_count"],
                    })
    if comparison_rows:
        before_after_df = pd.DataFrame(comparison_rows)
        print("\n=== BẢNG ĐỐI CHIẾU TRƯỚC VÀ SAU TỐI ƯU ===")
        display(before_after_df)
        before_after_df.to_csv(DATA / "comparison-before-after.csv", index=False)
else:
    aggregate = pd.DataFrame()

## 4. Biểu đồ So sánh Đa chỉ số giữa các Chính sách Co giãn

Mỗi biểu đồ tuân thủ quy chuẩn của đề cương:
- Có **tên trục** và **đơn vị đo** rõ ràng (`ms`, `%`, `Replica-seconds / 1k req`, `seconds`, `count`).
- Tiêu đề phụ (subtitle) ghi rõ **điều kiện đo** (`Endpoint`, `BaseRate`, `MaxVUs`, `WorkTime`/`Iterations`, `Step`) và **số lần lặp** ($n$) của từng chính sách.

In [ ]:
if not aggregate.empty:
    for (scenario, endpoint), subset in aggregate.groupby(["scenario", "endpoint"]):
        cond_str = format_condition_str(subset)
        repeats_str = ", ".join(f"{r.policy}: n={int(r.repeats)}" for _, r in subset.iterrows())
        labels = [f"{POLICY_LABELS.get(p, p)}\n(n={int(n)})" for p, n in zip(subset["policy"], subset["repeats"])]
        colors = [POLICY_COLORS.get(p, "#4C72B0") for p in subset["policy"]]
        x = np.arange(len(subset))

        # 1. Standalone p95 latency chart (for direct inclusion in report)
        fig, ax = plt.subplots(figsize=(9, 5.2))
        yerr = subset["p95_ms_std"].fillna(0).to_numpy() if (subset["repeats"] > 1).any() else None
        bars = ax.bar(x, subset["p95_ms"], yerr=yerr, capsize=5, color=colors, edgecolor="black", linewidth=0.7, width=0.55)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("p95 End-to-End Latency (ms)")
        ax.set_xlabel("Chính sách tự động co giãn (Autoscaling Policy)")
        ax.set_title(
            f"So sánh Độ trễ p95 — Kịch bản: {scenario.upper()} ({endpoint.upper()})\n"
            f"Điều kiện đo: {cond_str} | Số lần lặp: {repeats_str}",
            fontsize=10.5,
            pad=12,
        )
        for bar, val in zip(bars, subset["p95_ms"]):
            if pd.notna(val):
                ax.annotate(
                    f"{val:.1f} ms",
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 4),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                )
        fig.tight_layout()
        fname = f"p95-{scenario}-{endpoint}"
        fig.savefig(FIGURES / f"{fname}.png", dpi=200)
        fig.savefig(FIGURES / f"{fname}.pdf")
        plt.show()

        # 2. Multi-metric 2x2 tradeoff dashboard (QoS p95, Error Rate, Cost per 1k req, Scaling Dynamics)
        fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
        fig.suptitle(
            f"Đánh đổi Hiệu năng - Chi phí - Độ ổn định — Kịch bản: {scenario.upper()} ({endpoint.upper()})\n"
            f"Điều kiện đo: {cond_str} | Số lần lặp: {repeats_str}",
            fontsize=11.5,
            fontweight="bold",
        )

        # Panel (a): p95 Latency (ms)
        ax = axes[0, 0]
        ax.bar(x, subset["p95_ms"].fillna(0), color=colors, edgecolor="black", width=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("p95 Latency (ms)")
        ax.set_xlabel("Autoscaling Policy")
        ax.set_title("(a) Chất lượng dịch vụ: Độ trễ p95 (ms)", fontsize=10)

        # Panel (b): Error Rate (%)
        ax = axes[0, 1]
        ax.bar(x, subset["error_rate_pct"].fillna(0), color=colors, edgecolor="black", width=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("Error Rate (%)")
        ax.set_xlabel("Autoscaling Policy")
        ax.set_title("(b) Tỷ lệ lỗi yêu cầu (%)", fontsize=10)

        # Panel (c): Cost per 1,000 Successful Requests (Replica-seconds / 1k req)
        ax = axes[1, 0]
        ax.bar(x, subset["cost_per_1k_req_replica_s"].fillna(0), color=colors, edgecolor="black", width=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("Cost (Replica-seconds / 1k req)")
        ax.set_xlabel("Autoscaling Policy")
        ax.set_title("(c) Chi phí tài nguyên trên 1.000 yêu cầu thành công", fontsize=10)

        # Panel (d): Reaction Time (scale_out_s) & Replica Changes / Flapping
        ax = axes[1, 1]
        w = 0.25
        ax.bar(x - w, subset["scale_out_s"].fillna(0), width=w, label="Scale-out Reaction (s)", color="#4C72B0", edgecolor="black")
        ax.bar(x, subset["replica_changes"].fillna(0), width=w, label="Replica Changes (count)", color="#DD8452", edgecolor="black")
        ax.bar(x + w, subset["flapping_count"].fillna(0), width=w, label="Flapping Reversals (count)", color="#C44E52", edgecolor="black")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("Seconds (s) / Event Count (events)")
        ax.set_xlabel("Autoscaling Policy")
        ax.set_title("(d) Thời gian phản ứng Scale-out & Số lần dao động", fontsize=10)
        ax.legend(fontsize=8)

        fig.tight_layout(rect=[0, 0, 1, 0.93])
        tradeoff_name = f"tradeoff-{scenario}-{endpoint}"
        fig.savefig(FIGURES / f"{tradeoff_name}.png", dpi=200)
        fig.savefig(FIGURES / f"{tradeoff_name}.pdf")
        plt.show()
else:
    print("Chưa có dữ liệu tổng hợp để vẽ biểu đồ so sánh.")

## 5. Biểu đồ Đồng bộ theo Thời gian (Tải đầu vào — Hàng đợi / Bão hòa — Số bản sao & CPU)

Đáp ứng yêu cầu minh chứng: *"Thể hiện đồng bộ tải, số bản sao và độ trễ/hàng đợi theo thời gian"* kèm đầy đủ nhãn trục, đơn vị, điều kiện đo và lần lặp ($r$).

In [ ]:
if not runs.empty:
    for meta_file in sorted(DATA.glob("*/*/metadata.json")):
        timeline = meta_file.parent / "prometheus_timeseries.csv"
        if not timeline.exists():
            continue
        df = pd.read_csv(timeline)
        if df.empty:
            continue
        meta = json.loads(meta_file.read_text(encoding="utf-8-sig"))
        for col in ["timestamp", "app_rps", "in_flight", "queue_length", "replicas_available", "replicas_desired", "cpu_cores"]:
            if col in df:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.sort_values("timestamp").reset_index(drop=True)

        rep_col = next((k for k in ("replicas_available", "replicas_desired") if k in df and df[k].notna().any()), None)
        if rep_col is None:
            continue

        t0 = float(meta.get("started_unix", df["timestamp"].iloc[0]))
        t_end = float(meta.get("ended_unix", df["timestamp"].iloc[-1]))
        rel_t = df["timestamp"] - t0

        policy = meta.get("policy", meta_file.parent.parent.name)
        scenario = meta.get("scenario", "unknown")
        endpoint = meta.get("endpoint", "unknown")
        repeat_no = int(meta.get("repeat", 1))
        rate_val = meta.get("rate", "?")
        vus_val = meta.get("max_vus", "?")
        step_val = meta.get("sample_seconds", 5)
        extra_cond = (
            f"Iterations={meta.get('iterations')}"
            if endpoint == "cpu"
            else f"WorkTime={meta.get('work_time')}s, QueueLimit={meta.get('max_queue_limit', meta.get('queue_limit'))}"
        )
        subtitle = (
            f"Policy={policy} | Scenario={scenario} ({endpoint}) | Repeat r={repeat_no} (n=1 run timeline)\n"
            f"Điều kiện đo: BaseRate={rate_val} req/s, MaxVUs={vus_val}, {extra_cond}, Step={step_val}s"
        )

        fig, axes = plt.subplots(3, 1, figsize=(11, 8.2), sharex=True)
        fig.suptitle(f"Diễn biến Đồng bộ Tải - Hàng đợi - Bản sao: {meta_file.parent.name}\n{subtitle}", fontsize=10.5, fontweight="bold")

        # Panel 1: Load (RPS & In-Flight Requests)
        ax1 = axes[0]
        if "app_rps" in df and df["app_rps"].notna().any():
            ax1.plot(rel_t, df["app_rps"], color="#1f77b4", linewidth=2, label="Throughput (req/s)")
        if "in_flight" in df and df["in_flight"].notna().any():
            ax1.plot(rel_t, df["in_flight"], color="#ff7f0e", linewidth=1.8, linestyle="--", label="In-Flight Requests (count)")
        ax1.axvline(0, color="gray", linestyle=":", linewidth=1, label="Workload Start (t=0s)")
        ax1.axvline(t_end - t0, color="black", linestyle=":", linewidth=1, label="Workload End")
        ax1.set_ylabel("Load (req/s & count)")
        ax1.set_title("(1) Tải đầu vào (Throughput RPS) & Yêu cầu đang xử lý (In-Flight)", fontsize=9.5)
        ax1.legend(loc="upper right", fontsize=8)

        # Panel 2: Queue Backlog (Queue Length)
        ax2 = axes[1]
        if "queue_length" in df and df["queue_length"].notna().any():
            ax2.plot(rel_t, df["queue_length"], color="#d62728", linewidth=2, label="Queue Length (waiting requests)")
            ax2.fill_between(rel_t, 0, df["queue_length"].fillna(0), color="#d62728", alpha=0.15)
        ax2.axvline(0, color="gray", linestyle=":", linewidth=1)
        ax2.axvline(t_end - t0, color="black", linestyle=":", linewidth=1)
        ax2.set_ylabel("Queue Length (requests)")
        ax2.set_title("(2) Chỉ số Nghiệp vụ & Độ nghẽn: Số yêu cầu chờ trong Hàng đợi (Queue Length)", fontsize=9.5)
        ax2.legend(loc="upper right", fontsize=8)

        # Panel 3: Replicas & CPU Usage
        ax3 = axes[2]
        ax3.step(rel_t, df[rep_col], where="post", color="#2ca02c", linewidth=2.5, label=f"Active Replicas ({rep_col})")
        if "replicas_desired" in df and rep_col != "replicas_desired" and df["replicas_desired"].notna().any():
            ax3.step(rel_t, df["replicas_desired"], where="post", color="#98df8a", linestyle="--", linewidth=1.5, label="Desired Replicas")
        ax3.set_ylabel("Replicas (pods)")
        ax3.set_xlabel("Thời gian tương đối tính từ lúc bắt đầu kịch bản tải (Elapsed Time, seconds)")
        ax3.set_title("(3) Động học Co giãn Bản sao (Pod Replicas) & Mức sử dụng CPU (cores)", fontsize=9.5)

        if "cpu_cores" in df and df["cpu_cores"].notna().any():
            ax3_cpu = ax3.twinx()
            ax3_cpu.plot(rel_t, df["cpu_cores"], color="#9467bd", linewidth=1.5, alpha=0.75, label="Total CPU Usage (cores)")
            ax3_cpu.set_ylabel("CPU Usage (cores)", color="#9467bd")
            ax3_cpu.grid(False)
            lines1, labels1 = ax3.get_legend_handles_labels()
            lines2, labels2 = ax3_cpu.get_legend_handles_labels()
            ax3.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
        else:
            ax3.legend(loc="upper right", fontsize=8)

        ax3.axvline(0, color="gray", linestyle=":", linewidth=1)
        ax3.axvline(t_end - t0, color="black", linestyle=":", linewidth=1)

        fig.tight_layout(rect=[0, 0, 1, 0.92])
        fig.savefig(FIGURES / f"replicas-{meta_file.parent.name}.png", dpi=200)
        fig.savefig(FIGURES / f"replicas-{meta_file.parent.name}.pdf")
        plt.close(fig)

    print("Đã xuất toàn bộ biểu đồ ra thư mục:", FIGURES)
else:
    print("Chưa có lượt chạy nào để xuất biểu đồ timeline.")

## 6. Tổng kết Phân tích & Giới hạn Phương pháp

### Q&A
- **Câu hỏi nghiên cứu:** *Chính sách co giãn theo CPU có đủ hiệu quả đối với tải biến động nhanh, hay cần kết hợp độ trễ, hàng đợi và số yêu cầu đang chờ?*
- **Phương pháp kiểm chứng trong Notebook:** Đối chiếu trực tiếp 3 chính sách (`baseline-cpu`, `custom-unoptimized`, `custom-optimized`) trên cùng điều kiện tải thông qua độ trễ `p95_ms`, tỷ lệ lỗi `error_rate_pct` (và lỗi quá tải HTTP 503), thời gian phản ứng `scale_out_s`, chi phí `cost_per_1k_req_replica_s` (`ReplicaSeconds` trên 1.000 yêu cầu thành công) và số lần dao động đảo chiều `flapping_count`.

### Data Analysis Key Findings
- Với công việc xử lý chờ theo hàng đợi (`/api/queue`), mức tiêu thụ CPU tăng rất ít ngay cả khi cả 5 worker của Pod đã bão hòa 100%, khiến HPA chỉ dựa trên CPU không mở rộng kịp thời khi hàng đợi tích tụ.
- Chỉ số `ReplicaSeconds` và `cost_per_1k_req_replica_s` cho phép định lượng trực tiếp sự đánh đổi giữa chất lượng dịch vụ (`p95_ms`, `error_rate_pct`) và chi phí tài nguyên cấp phát theo thời gian.
- Tách biệt `replica_changes` (tổng số bước thay đổi bản sao) và `flapping_count` (số lần đảo chiều tăng $\leftrightarrow$ giảm trong cửa sổ $\le 90\text{s}$) giúp đánh giá chính xác hiệu quả của cửa sổ ổn định (`stabilizationWindowSeconds: 60`) trong kịch bản tải Chu kỳ (`periodic`).

### Insights or Next Steps
- Chạy đầy đủ ma trận thực nghiệm cho cả 3 chính sách (`baseline-cpu`, `custom-unoptimized`, `custom-optimized`) trên 3 kịch bản tải (`step`, `spike`, `periodic`) và kịch bản lỗi (`fault`), sau đó chạy lại toàn bộ các ô trong Notebook này để cập nhật bảng `data/comparison-before-after.csv` và bộ biểu đồ trong `notebooks/figures/`.
- Lưu ý giới hạn phương pháp: kết quả đo trên cụm Minikube 1 node phản ánh chính xác hành vi điều khiển của HPA/KEDA và hiệu quả chống dao động, nhưng chưa bao gồm thời gian cấp phát node mới (Cluster Autoscaler) trên môi trường đám mây đa node.